# Model Training

## Imports

In [1]:
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic
import torch
import os
import random
import shutil
from pathlib import Path

## Check GPU availability

In [2]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU
GPU count: 1


## Splitting into train/valid/test folders

In [15]:
dataset_root = Path("PAPI_Night.yolo26")

random.seed(42)


def find_images_and_labels(root):
    # case 1: structured dataset
    if (root / "train" / "images").exists():
        img_dir = root / "train" / "images"
        lbl_dir = root / "train" / "labels"
    else:
        # fallback: flat train folder
        img_dir = root / "train"
        lbl_dir = root / "train"

    return img_dir, lbl_dir


def get_pairs(img_dir, lbl_dir):
    exts = [".jpg", ".jpeg", ".png"]

    label_map = {p.stem.lower(): p for p in lbl_dir.glob("*.txt")}
    pairs = []

    for img in img_dir.iterdir():
        if img.suffix.lower() in exts:
            key = img.stem.lower()

            if key in label_map:
                pairs.append((img, label_map[key]))

    return pairs


def split(pairs):
    random.shuffle(pairs)

    n = len(pairs)
    train_end = int(n * 0.7)
    val_end = train_end + int(n * 0.2)

    return (
        pairs[:train_end],
        pairs[train_end:val_end],
        pairs[val_end:]
    )


def move(pairs, split_name, root):
    img_out = root / split_name / "images"
    lbl_out = root / split_name / "labels"

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img, lbl in pairs:
        shutil.move(str(img), img_out / img.name)
        shutil.move(str(lbl), lbl_out / lbl.name)


def run():
    img_dir, lbl_dir = find_images_and_labels(dataset_root)

    print("Using:")
    print("Images:", img_dir)
    print("Labels:", lbl_dir)

    pairs = get_pairs(img_dir, lbl_dir)

    if not pairs:
        raise ValueError("No pairs found — check dataset structure")

    train, val, test = split(pairs)

    move(train, "train", dataset_root)
    move(val, "valid", dataset_root)
    move(test, "test", dataset_root)

    print("Total:", len(pairs))
    print("Train:", len(train))
    print("Val:", len(val))
    print("Test:", len(test))


run()

Using:
Images: PAPI_Night.yolo26\train\images
Labels: PAPI_Night.yolo26\train\labels
Total: 626
Train: 438
Val: 125
Test: 63


## Model training

In [16]:
def train():
    model = YOLO("yolo26n.pt")

    model.train(
        data="PAPI_Night.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True
    )

train()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=PAPI_Night.yolo26\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

In [18]:
def evaluate():
    model = YOLO("runs/detect/train-2/weights/best.pt")

    metrics = model.val(data="PAPI_Night.yolo26\data.yaml")

    print(metrics)

evaluate()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 989.3339.1 MB/s, size: 2844.3 KB)
val: Scanning D:\School\Howest\Industry Project\Git\data\PAPI_Night.yolo26\valid\labels.cache... 125 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 125/125  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.2ss
                   all        125        500      0.706      0.837      0.786      0.367
              PAPI-Red        104        265      0.603      0.755      0.665      0.286
            PAPI-White        104        235      0.809      0.919      0.906      0.448
Speed: 2.6ms preprocess, 6.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to D:\School\Howest\Industry Project\Git\data\runs\detect\val
ultralytics.ut

In [19]:
def export():
    model = YOLO("runs/detect/train-2/weights/best.pt")

    model.export(
        format="onnx",
        imgsz=640,
        dynamic=True
    )

export()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CPU (13th Gen Intel Core i7-13650HX)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'runs\detect\train-2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "c:\Users\maxim\anaconda3\python.exe" "onnxslim>=0.1.71" "onnxruntime-gpu"  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "c:\Users\maxim\anaconda3\python.exe" "onnxslim>=0.1.71" "onnxruntime-gpu"  --index-strategy=unsafe-best-match --break-

c:\Users\maxim\anaconda3\Lib\site-packages\torch\onnx\symbolic_opset9.py:5385: UserWarning: Exporting aten::index operator of advanced indexing in opset 19 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success  209.7s, saved as 'runs\detect\train-2\weights\best.onnx' (9.5 MB)

Export complete (210.0s)
Results saved to D:\School\Howest\Industry Project\Git\data\runs\detect\train-2\weights\best.onnx
Predict:         yolo predict task=detect model=runs\detect\train-2\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs\detect\train-2\weights\best.onnx imgsz=640 data=PAPI_Night.yolo26\data.yaml  
Visualize:       https://netron.app


## Edge optimization

In [21]:
def quantize_model():
    quantize_dynamic(
        model_input=r"runs\detect\train-2\weights\best.onnx",
        model_output="best_int8.onnx",
        weight_type=3  # INT8
    )

quantize_model()

## Training YOLO26s (s=small) model for comparison

In [4]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Night.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True
    )

train()

New https://pypi.org/project/ultralytics/8.4.52 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=PAPI_Night.yolo26\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-3/weights/best.pt")

    metrics = model.val(data="PAPI_Night.yolo26\data.yaml")

    print(metrics)

evaluate()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2734.5620.2 MB/s, size: 1181.6 KB)
val: Scanning D:\School\Howest\Industry Project\Git data\data\PAPI_Night.yolo26\valid\labels.cache... 125 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 125/125  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.0it/s 4.1s0.2ss
                   all        125        500      0.661      0.806      0.756      0.383
              PAPI-Red        104        265      0.515      0.679      0.582      0.287
            PAPI-White        104        235      0.806      0.932      0.929      0.478
Speed: 2.9ms preprocess, 8.2ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to D:\School\Howest\Industry Project\Git data\data\runs\detect\val-2


## Training on augmented dataset

In [4]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Night_Augmented.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True
    )

train()

New https://pypi.org/project/ultralytics/8.4.52 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=PAPI_Night_Augmented.yolo26\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, mu

In [5]:
def evaluate():
    model = YOLO("runs/detect/train-5/weights/best.pt")

    metrics = model.val(data="PAPI_Night_Augmented.yolo26\data.yaml")

    print(metrics)

evaluate()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1061.9906.4 MB/s, size: 3358.5 KB)
val: Scanning D:\School\Howest\Industry Project\Git data\data\PAPI_Night_Augmented.yolo26\valid\labels.cache... 375 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 375/375  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.9it/s 8.4s0.2s
                   all        375       1500      0.743      0.776       0.76      0.348
              PAPI-Red        312        795      0.652      0.673      0.641      0.262
            PAPI-White        312        705      0.834      0.879      0.878      0.434
Speed: 1.8ms preprocess, 5.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to D:\School\Howest\Industry Project\Git data\data\runs\de

## Training on full dataset

In [2]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Split\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True
    )

train()

New https://pypi.org/project/ultralytics/8.4.54 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=PAPI_Split\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

In [3]:
def evaluate():
    model = YOLO("runs/detect/train-6/weights/best.pt")

    metrics = model.val(data="PAPI_Split\data.yaml")

    print(metrics)

evaluate()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2383.2553.0 MB/s, size: 3256.0 KB)
val: Scanning D:\School\Howest\Industry Project\Git data\data\PAPI_Split\valid\labels.cache... 638 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 638/638  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 3.1it/s 12.9s0.2s
                   all        638       2547      0.919      0.851      0.939      0.646
              PAPI-Red        559       1371      0.892      0.826      0.918      0.629
            PAPI-White        479       1176      0.946      0.876      0.961      0.663
Speed: 1.5ms preprocess, 5.1ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to D:\School\Howest\Industry Project\Git data\data\runs\detect\val-4
ultra

## Training 1280x1280 model instead of 640x640 for comparison

In [4]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Split\data.yaml",
        epochs=100,
        imgsz=1280,
        batch=4,
        device=0,
        workers=1,
        patience=15,
        save=True
    )

train()

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=PAPI_Split\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

In [5]:
def evaluate():
    model = YOLO("runs/detect/train-7/weights/best.pt")

    metrics = model.val(data="PAPI_Split\data.yaml")

    print(metrics)

evaluate()

Ultralytics 8.4.51  Python-3.11.7 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2392.8263.4 MB/s, size: 3256.0 KB)
val: Scanning D:\School\Howest\Industry Project\Git data\data\PAPI_Split\valid\labels.cache... 638 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 638/638  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 1.6it/s 24.9s0.5ss
                   all        638       2547      0.949      0.937      0.983      0.715
              PAPI-Red        559       1371      0.937      0.913      0.977      0.707
            PAPI-White        479       1176      0.961      0.962      0.989      0.722
Speed: 8.7ms preprocess, 20.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to D:\School\Howest\Industry Project\Git data\data\runs\detect\val-5
ult